# Data complexity effects on synthetic data quality

## Step 3: Evaluate synthetic data

This notebook reads the real and synthetic manifests and evaluates saved synthetic CSVs in terms of their complexity, quality, privacy, and utility.

## Load libraries

In [ ]:
!pip install pymdma
!pip install sdv
!pip install problexity

In [ ]:
import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from torch import manual_seed
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sdv.evaluation.single_table import run_diagnostic, evaluate_quality
from sdv.metadata import Metadata
from problexity import ComplexityCalculator


In [ ]:
try:
  from pymdma.tabular.measures.synthesis_val import ImprovedPrecision, ImprovedRecall, DCRPrivacy, Authenticity, Coverage, Density, CoherenceScore
  PYMDMA_AVAILABLE = True
except ImportError:
  PYMDMA_AVAILABLE = False


In [ ]:
EXP = "syndaite"

PROJECT_DIR = Path.cwd().resolve()

def project_path(path):
  path = Path(path)
  return path if path.is_absolute() else PROJECT_DIR / path

INPUT_DIR = PROJECT_DIR / "data" / "synth_input" / EXP
OUTPUT_DIR = PROJECT_DIR / "data" / "synth_output" / EXP
RESULT_DIR = PROJECT_DIR / "results" / EXP
FIG_DIR = PROJECT_DIR / "results" / "figs" / EXP

RANDOM_SEEDS = [42]
test_size = 0.33
K = 5 # pymdma default value for k-nearest neighbors


In [ ]:
# Set seeds
np.random.seed(RANDOM_SEEDS[0])
manual_seed(RANDOM_SEEDS[0])
random.seed(RANDOM_SEEDS[0])


In [ ]:
def get_feature_cols(df, target_col):
  return [c for c in df.columns if c != target_col]


def fix_target_dtype(df, target_col):
  df[target_col] = df[target_col].astype("object")
  return(df)


def coerce_target(y):
  return pd.to_numeric(y).round().astype(int).to_numpy()


## Load manifests

In [ ]:
manifest = pd.read_csv(INPUT_DIR / f"{EXP}_manifest.csv")
manifest


In [ ]:
synthetic_manifest = pd.read_csv(OUTPUT_DIR / f"{EXP}_synthetic_manifest.csv")
synthetic_manifest


## Evaluation helpers

In [ ]:
comp_scores = ['f1', 'f1v', 'f2', 'f3', 'f4', 'l1', 'l2', 'l3', 'n1', 'n2', 'n3', 'n4', 't1', 'lsc', 'density', 'clsCoef', 'hubs', 't2', 't3', 't4', 'c1', 'c2']


evals = {
    "sdv_diag": run_diagnostic,
    "sdv_qual": evaluate_quality
}

if PYMDMA_AVAILABLE:
  metrics = {
      "Privacy": DCRPrivacy(),
      "Authenticity": Authenticity(),
      "ImpPrecision": ImprovedPrecision(k=K, metric="euclidean"),
      "ImpRecall": ImprovedRecall(k=K, metric="euclidean"),
      "Density" : Density(k=K, metric="euclidean"),
      "Coverage" : Coverage(k=K, metric="euclidean"),
      "Coherence" : CoherenceScore()
  }
else:
  metrics = {}


def add_sdv_rows(res, report, dataset_name, syn_name, rep_name, eval_name):
  props = pd.DataFrame(report.get_properties())
  for _, row in props.iterrows():
    res.append({
        "Dataset": dataset_name,
        "Synthesizer": syn_name,
        "Repetition": rep_name,
        "EvalType": eval_name,
        "Property": row["Property"],
        "Score": row["Score"],
    })


def add_report_rows(res, acc, f1, dataset_name, syn_name, rep_name, model_name, eval_name):
  res.append({
      "Dataset": dataset_name,
      "Synthesizer": syn_name,
      "Repetition": rep_name,
      "Classifier": model_name,
      "EvalType": eval_name,
      "Property": "accuracy",
      "Score": acc,
  })
  res.append({
      "Dataset": dataset_name,
      "Synthesizer": syn_name,
      "Repetition": rep_name,
      "Classifier": model_name,
      "EvalType": eval_name,
      "Property": "macro_f1",
      "Score": f1,
  })

def add_score_row(res, property, score, dataset_name, syn_name, rep_name, eval_name):
  res.append({
      "Dataset": dataset_name,
      "Synthesizer": syn_name,
      "Repetition": rep_name,
      "EvalType": eval_name,
      "Property": property,
      "Score": score,
  })


In [ ]:
runtime_warnings = []

warnings.filterwarnings(
    "ignore",
    message=r".*Downcasting object dtype arrays on \.fillna.*",
    category=FutureWarning,
)


def complexity_report_with_warning_log(X, y, dataset_name, synth_name=None, rep_name=None, stage=None):
  with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    cc = ComplexityCalculator().fit(X, y)
    report = cc.report()

  for warning in caught:
    runtime_warnings.append({
        "Dataset": dataset_name,
        "Synthesizer": synth_name,
        "Repetition": rep_name,
        "Stage": stage,
        "Category": warning.category.__name__,
        "Message": str(warning.message),
        "Filename": warning.filename,
        "Line": warning.lineno,
    })
  return report


## Evaluate quality, privacy, utility, and complexity

In [ ]:
# %%capture --no-stdout

RESULT_DIR.mkdir(parents=True, exist_ok=True)

res = []  
input_complex = []
base_res = []
manifest_by_dataset = manifest.set_index("Dataset")

base_ratios = []
ratios = []

i = 0
for dataset_name, syn_rows in synthetic_manifest.groupby("Dataset"):

  if(i % 100 == 0): 
    print(dataset_name)

  # read data
  ds_info = manifest_by_dataset.loc[dataset_name]
  target = ds_info["target_col"]
  train = fix_target_dtype(pd.read_csv(project_path(ds_info.train_path)), target_col=target)
  test = fix_target_dtype(pd.read_csv(project_path(ds_info.test_path)), target_col=target)
  
  # get feature columns and metadata
  feature_cols = get_feature_cols(train, target_col=target)
  meta1 = Metadata.load_from_json(str(project_path(ds_info.metadata_path)))
  
  # divide into X and y
  X_train = train[feature_cols].astype(float)
  y_train = coerce_target(train[target])
  X_test = test[feature_cols].astype(float)
  y_test = coerce_target(test[target])

  # save imbalance ratios for train and test sets
  base_ratios.append({"Dataset": dataset_name,
                "Train_IR" : train[[target]].value_counts().min() / train[[target]].value_counts().max(),
                "Test_IR" : test[[target]].value_counts().min() / test[[target]].value_counts().max()})

  # calculate complexity of the input data
  input_comp_vals = complexity_report_with_warning_log(X_train, y_train, dataset_name, stage="input")
  input_comp_vals["complexities"] = {'input_' + k: v for k, v in input_comp_vals["complexities"].items()}
  input_complex.append({"Dataset": dataset_name, "input_complex": input_comp_vals["score"]} | input_comp_vals["complexities"])

  # scale the data for logistic regression
  scalerd = StandardScaler().fit(X_train)
  real_scaled = scalerd.transform(X_train)
  test_scaled_real = scalerd.transform(X_test)

  models = {
    "LR": LogisticRegression(max_iter=500),
    "DT": DecisionTreeClassifier(random_state=RANDOM_SEEDS[0]),
    "RF": RandomForestClassifier(random_state=RANDOM_SEEDS[0]),
  }

  # evaluate train on real, test on real
  for model_name, model in models.items():
    if model_name == "LR":
      model.fit(real_scaled, y_train)
      y_pred_test = model.predict(test_scaled_real)
    else:
      model.fit(X_train, y_train)
      y_pred_test = model.predict(X_test)
    util_report = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
    add_report_rows(base_res, util_report["accuracy"], util_report["macro avg"]["f1-score"], dataset_name, None, None, model_name, "base_utility")

  for syn_info in syn_rows.itertuples(index=False):

    # read synthetic data
    syn_df = fix_target_dtype(pd.read_csv(project_path(syn_info.synthetic_path)), target_col=target)
    X_syn = syn_df[feature_cols].astype(float)
    y_syn = coerce_target(syn_df[target])

    only_one_class = len(np.unique(y_syn)) == 1
    wrong_classes = not set(np.unique(y_syn)).issubset(set(np.unique(y_train)))  

    if(only_one_class or wrong_classes):
      if(only_one_class):
        runtime_warnings.append({
                "Dataset": dataset_name,
                "Synthesizer": syn_info.Model,
                "Repetition": syn_info.Repetition,
                "Stage": "generation",
                "Category": "OnlyOneClassWarning",
                "Message": "Synthetic data has only one class in the target variable.",
                "Filename": None,
                "Line": None,
            })
      if(wrong_classes):
        runtime_warnings.append({
                "Dataset": dataset_name,
                "Synthesizer": syn_info.Model,
                "Repetition": syn_info.Repetition,
                "Stage": "generation",
                "Category": "WrongClassesWarning",
                "Message": "Synthetic data has classes not present in the real training data.",
                "Filename": None,
                "Line": None,
            })
      # save synthetic imbalance ratio
      ratios.append({"Dataset": dataset_name, 
                    "Synthesizer": syn_info.Model, 
                    "Repetition": syn_info.Repetition, 
                    "Output_IR": pd.NA})
    else:
      # save synthetic imbalance ratio
      ratios.append({"Dataset": dataset_name, 
                    "Synthesizer": syn_info.Model, 
                    "Repetition": syn_info.Repetition, 
                    "Output_IR": syn_df[[target]].value_counts().min() / syn_df[[target]].value_counts().max()})

    # calculate sdv evaluation metrics
    for eval_name, eval_fun in evals.items():
      report = eval_fun(real_data=train, synthetic_data=syn_df, metadata=meta1, verbose=False)
      add_sdv_rows(res, report, dataset_name, syn_info.Model, syn_info.Repetition, eval_name)

    # calculate pymdma evaluation metrics
    syn_scaled = scalerd.transform(X_syn) # scale using real data for better comparison
    if metrics:
      for metric_name, metric in metrics.items():
        result = metric.compute(real_scaled, syn_scaled)
        if metric_name == "Privacy":
          dataset_level = result.value[0]['privacy'] / 100
        else:
          dataset_level, _ = result.value
        res.append({"Dataset": dataset_name, 
                    "Synthesizer": syn_info.Model, 
                    "Repetition": syn_info.Repetition, 
                    "EvalType": "pymdma",
                    "Property": metric_name, 
                    "Score": dataset_level})

    # scale the test data based on synthetic training data for LR
    scalerd = StandardScaler().fit(X_syn)
    syn_scaled = scalerd.transform(X_syn) # scale using synthetic data only to prevent leakage
    test_scaled_syn = scalerd.transform(X_test)

    if not (only_one_class or wrong_classes):
      # evaluate train on synthetic, test on real
      for model_name, model in models.items():
        if model_name == "LR":
          model.fit(syn_scaled, y_syn)
          y_pred_test = model.predict(test_scaled_syn)
        else:
          model.fit(X_syn, y_syn)
          y_pred_test = model.predict(X_test)
        util_report = classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)
        add_report_rows(res, util_report["accuracy"], util_report["macro avg"]["f1-score"], 
                        dataset_name, syn_info.Model, syn_info.Repetition, model_name, "utility")

      # calculate complexity of the synthetic data
      comp_vals = complexity_report_with_warning_log(
          X_syn,
          y_syn,
          dataset_name,
          synth_name=syn_info.Model,
          rep_name=syn_info.Repetition,
          stage="output",
      )
      comp_vals["complexities"] = {'output_' + k: v for k, v in comp_vals["complexities"].items()}
      for k, v in comp_vals["complexities"].items(): 
        add_score_row(res, k, v, dataset_name, syn_info.Model, syn_info.Repetition, "problexity")
      add_score_row(res, "output_complex", comp_vals["score"], dataset_name, syn_info.Model, syn_info.Repetition, "problexity")
    else:
      for model_name, model in models.items():
        add_report_rows(res, pd.NA, pd.NA, dataset_name, syn_info.Model, syn_info.Repetition, model_name, "utility")
      for k in comp_scores: 
        add_score_row(res, "output_" + k, pd.NA, dataset_name, syn_info.Model, syn_info.Repetition, "problexity")
      add_score_row(res, "output_complex", pd.NA, dataset_name, syn_info.Model, syn_info.Repetition, "problexity")

    i=i+1


In [ ]:
base_ratios = pd.DataFrame(base_ratios)
base_ratios["Diff"] = base_ratios["Test_IR"] - base_ratios["Train_IR"]
ratios = pd.DataFrame(ratios)
ratios = pd.merge(ratios, base_ratios, on="Dataset", how="left")


In [ ]:
eval_res = pd.DataFrame(res)
base_utility = pd.DataFrame(base_res)
input_complex_df = pd.DataFrame(input_complex)
warning_log = pd.DataFrame(runtime_warnings)

eval_res.head()

In [ ]:
eval_res.to_csv(RESULT_DIR / f"{EXP}_eval_results.csv", index=False)
base_utility.to_csv(RESULT_DIR / f"{EXP}_base_utility.csv", index=False)
input_complex_df.to_csv(RESULT_DIR / f"{EXP}_input_complex.csv", index=False)
ratios.to_csv(RESULT_DIR / f"{EXP}_ratios.csv", index=False)
warning_log.to_csv(RESULT_DIR / f"{EXP}_runtime_warnings.csv", index=False)